# Trabajo practico especial

**integrantes**:
+ Candela Belen, Buggarini.
+ Nubia, Larios.
+ Ignacio, Quiñoneros.

## Analisis exploratorio de los datos

In [14]:
import pandas as pd
from os import path, listdir

df_calendar=pd.read_csv("calendar.csv.gz")
df_listings = pd.read_csv("listings (1).csv.gz")

# chequeamos si los archivos están en el directorio donde estamos parados
lista_archivos = listdir("./")
print(f"El contenido de la carpeta es: {lista_archivos}")

El contenido de la carpeta es: ['.git', '.gitignore', 'Azucar', 'calendar.csv.gz', 'entrega.ipynb', 'Entregaf.ipynb', 'HipotesisCande.ipynb', 'listings (1).csv.gz', 'prueba1.ipynb', 'prueba2.ipynb', 'prueba3.ipynb', 'prueba4.ipynb', 'pruebanub.ipynb', 'README.md', 'requirements.txt', 'tabla_cruzada.png', 'TPE_CienciaDeDatos']


**implementación de operaciones de limpieza de la base**

Primero separamos el dataset original para realizar el pre procesamiento de los datos.

Prepocesamientos que se realizaron:
+ En primera instancia eliminamos todas las columnas que tengan en su totalidad nulos, estas estan casi en su totalidad a informacion del host.
+ Tranformamos las variables dictomicas `source`, `has_availability`, `host_is_superhost`, `host_has_profile_pic` y `host_identify_verified`.
+ Limpiamos de caracteres la variable `price`

In [15]:
import numpy as np

#copiamos el conjunto para preservarlo
preprocessed_dataset = df_listings.copy()

#borramos columnas con todos sus valores nulos
preprocessed_dataset = preprocessed_dataset.drop('neighborhood_overview', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_since', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_response_time', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_response_rate', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_acceptance_rate', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_thumbnail_url', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_neighbourhood', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_total_listings_count', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_verifications', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('neighbourhood', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('neighbourhood_group_cleansed', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('calendar_updated', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('instant_bookable', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('listing_url',axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_url',axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_profile_id',axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_profile_url',axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_name',axis=1)
preprocessed_dataset = preprocessed_dataset.drop('scrape_id',axis=1)
preprocessed_dataset = preprocessed_dataset.drop('picture_url',axis=1)
#preprocessed_dataset = preprocessed_dataset.drop('host_id',axis=1)
preprocessed_dataset = preprocessed_dataset.drop('license',axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_picture_url',axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_about',axis=1)

#transformacion de variables a dicotomicas
preprocessed_dataset ['source']= preprocessed_dataset['source'].map({"city scrape": True, "previous scrape": False})
preprocessed_dataset ['has_availability']= preprocessed_dataset['has_availability'].map({"t": True, "f": False}).fillna(False).astype(bool)
preprocessed_dataset ['host_is_superhost']= preprocessed_dataset['host_is_superhost'].map({"t": True, "f": False})
preprocessed_dataset ['host_has_profile_pic']= preprocessed_dataset['host_has_profile_pic'].map({"t": True, "f": False})
preprocessed_dataset ['host_identity_verified']= preprocessed_dataset['host_identity_verified'].map({"t": True, "f": False})



#Preguntar si es posible y como saber si es una tipo de MCAR, podemos formular hipotesis? o sea x host no dice los precios, en x zona no se suelen publicar
preprocessed_dataset['price'] = preprocessed_dataset['price'].str.replace(r'[$,]', '', regex=True).astype(float).replace("nan", np.nan)


#como vamos a utilizar la variable price para una de nuestras hipotesis eliminamos los valores nulos, ya que no son demasiados y para el analisis que necesitamos hacer no podemos
#reemplazarlos por 0 o por la media
preprocessed_dataset.dropna(subset=['beds'],inplace=True)


In [16]:

#Queremos ver el tipo de alojamiento al que pertenece y vamos a imputar por la mediana de los precios de ese tipo de alojamiento para no tener nulos y evitar eliminar filas

# Cantidad y porcentaje de nulos
print(preprocessed_dataset["price"].isna().sum())
print(preprocessed_dataset["price"].isna().mean() * 100)
nulos = preprocessed_dataset[preprocessed_dataset["price"].isna()]

# Nulos por tipo de alojamiento
print(preprocessed_dataset.groupby("room_type")["price"].apply(lambda s: s.isna().sum()))
preprocessed_dataset["price"] = preprocessed_dataset["price"].fillna(
    preprocessed_dataset.groupby("room_type")["price"].transform("median")
)

preprocessed_dataset["price"].info()

465
1.7157405357538191
room_type
Entire home/apt    406
Hotel room           2
Private room        53
Shared room          4
Name: price, dtype: int64
<class 'pandas.Series'>
Index: 27102 entries, 0 to 29633
Series name: price
Non-Null Count  Dtype  
--------------  -----  
27102 non-null  float64
dtypes: float64(1)
memory usage: 423.5 KB


In [17]:
preprocessed_dataset['price'].describe()

count    2.710200e+04
mean     2.757155e+05
std      3.625632e+06
min      1.215500e+03
25%      7.620300e+04
50%      1.019891e+05
75%      1.402235e+05
max      1.716438e+08
Name: price, dtype: float64

In [18]:
preprocessed_dataset

,id,last_scraped,source,name,description,host_id,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,54019695,2026-07-02,True,Hermoso departamento con piscina en Puerto Mad...,Enjoy a stylish experience in this centrally-l...,298874610,6,9,6,8,...,NaN,NaN,NaN,NaN,NaN,24,24,0,0,NaN
1,41720307,2026-06-29,True,"Departamento dos ambientes, 2cuadras recoleta ...",Brand new remodeled apartment 2 blocks from th...,220409424,7,8,7,8,...,4.67,4.67,4.67,4.67,4.56,8,8,0,0,0.16
3,1482653002662185299,2026-07-02,True,Descanso ideal - Av. Sante Fe,"Large and comfortable apartment for 4 people, ...",283840529,6,10,3,6,...,5.00,4.95,4.95,4.89,4.89,12,12,0,0,1.78
4,1046394538968941382,2026-06-29,True,Excelente zona,Enjoy easy access to everything from this perf...,550970409,2,6,2,6,...,NaN,NaN,NaN,NaN,NaN,1,0,1,0,NaN
5,1708343918490071398,2026-06-29,True,Excelente departamento ubicado en Palermo soho,Surround yourself with style in this standout ...,696837138,1,1,0,0,...,NaN,NaN,NaN,NaN,NaN,1,1,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29629,1345285684048892167,2026-07-01,True,Casa amplia y luminosa con gran terraza y parr...,Spacious and luminous PH in Palermo Soho neigh...,675977429,1,5,1,5,...,4.75,4.95,5.00,4.95,5.00,1,1,0,0,1.32
29630,875989201115251661,2026-07-02,True,"Estudio en Nuñez, Puro Sol!","Beautiful studio, full sun in the Nuñez neighb...",36390,16,9,3,10,...,4.92,4.98,4.98,4.94,4.94,1,1,0,0,3.29
29631,17591997,2026-07-02,True,9 - Nice Loft private in Villa Crespo near Pal...,It is an old house with many apartments and ha...,22694851,11,8,11,8,...,4.74,4.94,4.88,4.78,4.79,17,15,2,0,3.06
29632,748623735416497161,2026-07-02,True,Oasis en la ciudad Habitación matrimonio al ja...,Beautiful room overlooking the garden and gall...,116120207,9,4,3,8,...,5.00,5.00,5.00,4.67,5.00,3,0,3,0,0.13


In [19]:
preprocessed_dataset['property_type'].value_counts()

property_type
Entire rental unit                    19591
Entire condo                           2599
Private room in rental unit             872
Entire serviced apartment               869
Private room in home                    646
Entire loft                             577
Entire home                             431
Room in hotel                           328
Private room in condo                   157
Entire vacation home                    138
Private room in guesthouse              118
Private room in bed and breakfast        98
Private room in casa particular          88
Private room in serviced apartment       57
Entire townhouse                         47
Room in boutique hotel                   41
Private room in townhouse                40
Private room in villa                    37
Shared room in home                      34
Shared room in guesthouse                33
Room in hostel                           29
Entire guesthouse                        24
Entire villa      

In [20]:
preprocessed_dataset['host_is_superhost'].info()

<class 'pandas.Series'>
Index: 27102 entries, 0 to 29633
Series name: host_is_superhost
Non-Null Count  Dtype
--------------  -----
27102 non-null  bool 
dtypes: bool(1)
memory usage: 238.2 KB


In [21]:
import pandas as pd # Buscar filas duplicadas en latitud, longitud y host\_id 
duplicados_exactos = preprocessed_dataset[preprocessed_dataset.duplicated(subset=['latitude', 'longitude', 'host_id'], keep=False) ] 
print(f'Listings duplicados por ubicación exacta: {len(duplicados_exactos)}')

Listings duplicados por ubicación exacta: 2486


In [22]:
duplicados_nombre = preprocessed_dataset[preprocessed_dataset.duplicated(subset=['name', 'host_id'], keep=False) ]
print(f'Listings duplicados por nombre: {len(duplicados_nombre)}')

Listings duplicados por nombre: 799


In [23]:
print(preprocessed_dataset['bathrooms'].describe())

count    25698.000000
mean         1.245661
std          0.706877
min          0.500000
25%          1.000000
50%          1.000000
75%          1.000000
max         22.000000
Name: bathrooms, dtype: float64



HIPOTESIS UNIVARIADA QUE HICIMOS
Precios por noche (los alquileres en argentina no bajan de $x), en la ciudad de Buenos Aires.


In [24]:

print(preprocessed_dataset['bathrooms'].value_counts())

#tenemos muchos outliers un minimo muy bajo y un maximo muy grande
#para definir la hipotesis, podemos filtrar? eliminamos outliers? planteamos la hipotesis con datos externos la media o la mediana?
#imputar por la mediana de oublicaciones del mismo tipo


bathrooms
1.0     19907
1.5      2530
2.0      1976
2.5       422
3.0       398
4.0        88
3.5        87
6.0        76
0.5        71
5.0        50
4.5        23
8.0        14
5.5        13
9.0        11
7.0        10
8.5         5
10.0        3
12.0        2
16.0        2
20.0        2
6.5         2
7.5         2
15.0        1
11.0        1
22.0        1
9.5         1
Name: count, dtype: int64


Analisis de la hipotesis Airbnb en Bs. As. es un negocio de grandes jugadores: más del 60% de la oferta la controlan agencias multipropiedad, no personas alquilando su propio
hogar.

calculated_host_listings_count                
calculated_host_listings_count_entire_homes    
calculated_host_listings_count_private_rooms
calculated_host_listings_count_shared_rooms 

Tengo varias dudas, como hacemos el manejo de por ejemplo las habitaciones de hotel, si un host tiene muchas podria ser un solo hotel, eso seria multipropiedad igual?

In [25]:
for k in [2, 3, 5, 10]:
    pct_hosts = (hosts[col] >= k).mean() * 100
    pct_oferta = (preprocessed_dataset[col] >= k).mean() * 100
    print(f'k={k:>2}: {pct_hosts:5.1f}% de los hosts | {pct_oferta:5.1f}% de la oferta')

NameError: name 'hosts' is not defined

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

col = "calculated_host_listings_count_entire_homes"
v = ent[col]

print(v.describe(percentiles=[.25, .5, .75, .90, .95, .99]))
print("Moda:", v.mode().iloc[0])
print("Asimetría:", v.skew())

plt.figure(figsize=(10, 3.5))
sns.boxplot(x=v)
plt.title("Boxplot de alojamientos enteros por anfitrión")
plt.xlabel("Alojamientos enteros del anfitrión (por publicación)")
plt.show()


p99 = v.quantile(0.99)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(v, discrete=True, ax=axes[0])
axes[0].set_title("Histograma completo")
axes[0].set_xlabel("Alojamientos enteros del anfitrión")

sns.histplot(v[v <= p99], discrete=True, ax=axes[1])
axes[1].axvline(v.median(), color="red", linestyle="--", label="mediana")
axes[1].axvline(v.mean(), color="green", linestyle="--", label="media")
axes[1].legend()
axes[1].set_title(f"Histograma recortado (hasta P99 = {p99:.0f})")
axes[1].set_xlabel("Alojamientos enteros del anfitrión")

plt.tight_layout()
plt.show()

In [ ]:
pct = ent["tramo"].value_counts(normalize=True).sort_index() * 100

plt.figure(figsize=(8, 4))
ax = pct.plot(kind="bar", color="steelblue")
ax.bar_label(ax.containers[0], fmt="%.1f%%")
plt.xticks(rotation=0)
plt.xlabel("Alojamientos enteros que tiene el anfitrión")
plt.ylabel("% de la oferta")
plt.title("Oferta de alojamientos enteros según tamaño del anfitrión")
plt.show()


#con este grafico ya nos vamos acercando al analisis de la hipotesis, pareciera ser verdad que el 60 son agencias multipropiedad

NameError: name 'ent' is not defined

A igual capacidad de huéspedes(accommodates), un alojamiento se alquila más caro (price) si
garantiza mayor independencia: tener una cama para cada persona (beds)(no
compartir colchón), mayor cantidad de baños por huésped (bathrooms), y reserva
automática.(es null)

In [ ]:
# 1) ¿Coincide con bathrooms donde ambas existen?
ambas = preprocessed_dataset["bathrooms"].notna() & preprocessed_dataset["banos_txt"].notna()
print("Coinciden:", (preprocessed_dataset.loc[ambas, "bathrooms"] == preprocessed_dataset.loc[ambas, "banos_txt"]).mean())
print(preprocessed_dataset.loc[ambas & (preprocessed_dataset["bathrooms"] != preprocessed_dataset["banos_txt"]),
            ["bathrooms", "bathrooms_text", "banos_txt"]].head(10))

# 2) ¿Hay textos que el parseo no entendió?
print(preprocessed_dataset.loc[preprocessed_dataset["banos_txt"].isna() & preprocessed_dataset["bathrooms_text"].notna(), "bathrooms_text"].unique())

# 3) Valores raros
print(preprocessed_dataset["banos_txt"].value_counts().sort_index().head(10))

Coinciden: 0.999847549355896
       bathrooms bathrooms_text  banos_txt
8589         1.0      1.5 baths        1.5
10237        1.5      2.5 baths        2.5
19202        1.0        0 baths        0.0
19295        1.0        0 baths        0.0
<StringArray>
[]
Length: 0, dtype: str
banos_txt
0.0      132
0.5       81
1.0    22897
1.5     2926
2.0     2244
2.5      486
3.0      451
3.5      101
4.0      109
4.5       24
Name: count, dtype: int64


In [ ]:
d = preprocessed_dataset.copy()

ambas = d["bathrooms"].notna() & d["banos_txt"].notna()
conflicto = ambas & (d["bathrooms"] != d["banos_txt"])
print("Conflictos:", conflicto.sum())
d = d[~conflicto]

Conflictos: 4


In [ ]:
print(d.loc[d["banos_txt"] == 0, "room_type"].value_counts())

# ¿Cuántos nulos de bathrooms se recuperaron con el texto?
print("bathrooms nulos:", d["bathrooms"].isna().sum())
print("recuperados con el texto:", (d["bathrooms"].isna() & d["banos_txt"].notna()).sum())
print("siguen nulos:", (d["bathrooms"].isna() & d["banos_txt"].isna()).sum())

room_type
Entire home/apt    99
Private room       26
Shared room         3
Hotel room          2
Name: count, dtype: int64
bathrooms nulos: 3441
recuperados con el texto: 3421
siguen nulos: 20


In [ ]:
n0 = len(d)
d["bathrooms_ok"] = d["bathrooms"].fillna(d["banos_txt"])

d = d.dropna(subset=["bathrooms_ok"])      # 20 sin dato en ninguna de las dos fuentes
n1 = len(d)
d = d[d["bathrooms_ok"] > 0]               # 132 con "0 baths": criterio desconocido
n2 = len(d)

print("Eliminados por baños nulos:", n0 - n1)
print("Eliminados por 0 baños:", n1 - n2)

NameError: name 'd' is not defined

In [ ]:
preprocessed_dataset["beds_nulo"] = preprocessed_dataset["beds"].isna()

print("% nulos:", round(preprocessed_dataset["beds_nulo"].mean() * 100, 1))
print(preprocessed_dataset.groupby("room_type")["beds_nulo"].mean().mul(100).round(1))
print(preprocessed_dataset.groupby("accommodates")["beds_nulo"].mean().mul(100).round(1).head(8))
print(preprocessed_dataset.groupby("beds_nulo")["price"].median())

% nulos: 0.0
room_type
Entire home/apt    0.0
Hotel room         0.0
Private room       0.0
Shared room        0.0
Name: beds_nulo, dtype: float64
accommodates
1    0.0
2    0.0
3    0.0
4    0.0
5    0.0
6    0.0
7    0.0
8    0.0
Name: beds_nulo, dtype: float64
beds_nulo
False    101989.06
Name: price, dtype: float64


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# df = tu dataset limpio

# 1. Elegir solo las variables de la hipótesis
y = preprocessed_dataset['price']
X = preprocessed_dataset[['accommodates', 'beds', 'banos_txt']]

# 2. Estandarizar solo los predictores
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)


KeyError: "['banos_txt'] not in index"

In [ ]:
# Imprimir las medias y desviaciones estándar antes del escalado
print("Medias y desviaciones estándar antes del escalado:")
print(pd.DataFrame(X).describe().loc[['mean', 'std']])

# Imprimir las medias y desviaciones estándar después del escalado
print("\nMedias y desviaciones estándar después del escalado:")
print(pd.DataFrame(X_scaled_df).describe().loc[['mean', 'std']])

Medias y desviaciones estándar antes del escalado:
      accommodates      beds  banos_txt
mean      2.941654  1.947753   1.236741
std       1.584278  1.621595   0.700423

Medias y desviaciones estándar después del escalado:
      accommodates          beds     banos_txt
mean  5.553172e-17  4.194777e-17  4.120616e-17
std   1.000017e+00  1.000018e+00  1.000017e+00


In [ ]:
print(X.isna().sum())
print(np.isinf(X).sum())
print("Nulos en y:", y.isna().sum())

accommodates       0
beds            2583
banos_txt         26
dtype: int64
accommodates    0
beds            0
banos_txt       0
dtype: int64
Nulos en y: 0


In [ ]:
# Agregamos una constante como característica, para estimar la ordenada al origen
import statsmodels.api as sm
X_scaled_df_ = sm.add_constant(X_scaled_df)
# Ajustamos el modelo
model = sm.OLS(y, X_scaled_df_).fit()
# Imprimimos el resumen de los resultados
print(model.summary())


